In [1]:
import os
import pandas as pd
import numpy as np
import kagglehub

In [2]:
# Download latest version
path = kagglehub.dataset_download("mexwell/smart-home-energy-consumption")

print("Path to dataset files:", path)

100%|██████████| 1.25M/1.25M [00:00<00:00, 1.34MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/mexwell/smart-home-energy-consumption/versions/1


In [9]:
import os

print(f"Listing contents of the downloaded directory: {path}")
for dirpath, dirnames, filenames in os.walk(path):
    for f in filenames:
        print(os.path.join(dirpath, f))


Listing contents of the downloaded directory: /root/.cache/kagglehub/datasets/mexwell/smart-home-energy-consumption/versions/1
/root/.cache/kagglehub/datasets/mexwell/smart-home-energy-consumption/versions/1/smart_home_energy_consumption_large.csv


In [14]:
import pandas as pd
import numpy as np
import os
import kagglehub
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans

# =========================
# DOWNLOAD DATASET
# =========================

path = kagglehub.dataset_download(
    "mexwell/smart-home-energy-consumption"
)

print("Dataset Path:", path)

# =========================
# FIND CSV
# =========================

csv_file = None

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".csv"):
            csv_file = os.path.join(root, file)

if csv_file is None:
    raise Exception("CSV not found")

print("CSV FILE:", csv_file)

# =========================
# LOAD DATA
# =========================

df = pd.read_csv(csv_file)

# =========================
# CLEANING
# =========================

df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# =========================
# TIME FEATURES
# =========================

df["Date"] = pd.to_datetime(df["Date"])
df["hour"] = pd.to_datetime(df["Time"], format="%H:%M").dt.hour

df["is_peak"] = df["hour"].apply(lambda x: 1 if 18 <= x <= 22 else 0)

# =========================
# ENCODING
# =========================

season_enc = LabelEncoder()
appliance_enc = LabelEncoder()

df["Season_enc"] = season_enc.fit_transform(df["Season"])
df["Appliance_enc"] = appliance_enc.fit_transform(df["Appliance Type"])

# =========================
# HOME PROFILING
# =========================

home_df = df.groupby("Home ID").agg({

    "Energy Consumption (kWh)": "mean",
    "Outdoor Temperature (°C)": "mean",
    "Household Size": "mean",
    "hour": "mean",
    "is_peak": "mean",
    "Season_enc": "mean",
    "Appliance_enc": "mean"

}).reset_index()

# =========================
# FEATURES
# =========================

features = [

    "Energy Consumption (kWh)",
    "Outdoor Temperature (°C)",
    "Household Size",
    "hour",
    "is_peak",
    "Season_enc",
    "Appliance_enc"

]

X = home_df[features]

# =========================
# SCALING
# =========================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# =========================
# KMEANS CLUSTERING
# =========================

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
home_df["cluster"] = kmeans.fit_predict(X_scaled)

# =========================
# SAVE
# =========================

joblib.dump(scaler, "scaler.pkl")
joblib.dump(kmeans, "kmeans.pkl")
home_df.to_csv("home_profiles.csv", index=False)

print("Training Complete")

Using Colab cache for faster access to the 'smart-home-energy-consumption' dataset.
Dataset Path: /kaggle/input/smart-home-energy-consumption
CSV FILE: /kaggle/input/smart-home-energy-consumption/smart_home_energy_consumption_large.csv
Training Complete


In [15]:
import pandas as pd

home_df = pd.read_csv("home_profiles.csv")

# =========================
# SOLAR PACKAGES
# =========================

SOLAR_PACKAGES = [

    {
        "name": "Small Home Kit",
        "max_kwh": 5,
        "panels": "2 x 550W Solar Panels",
        "battery": "5 kWh Lithium Battery",
        "inverter": "2 kW Hybrid Inverter"
    },

    {
        "name": "Medium Home Kit",
        "max_kwh": 10,
        "panels": "4 x 550W Solar Panels",
        "battery": "10 kWh Lithium Battery",
        "inverter": "3 kW Hybrid Inverter"
    },

    {
        "name": "Large Home Kit",
        "max_kwh": 20,
        "panels": "8 x 550W Solar Panels",
        "battery": "15 kWh Lithium Battery",
        "inverter": "5 kW Hybrid Inverter"
    },

    {
        "name": "Heavy Usage Kit",
        "max_kwh": 30,
        "panels": "12 x 550W Solar Panels",
        "battery": "20 kWh Lithium Battery",
        "inverter": "8 kW Hybrid Inverter"
    }

]

# =========================
# RECOMMENDER
# =========================

def recommend_solar(home_id):

    home = home_df[
        home_df["Home ID"] == home_id
    ]

    if len(home) == 0:
        return {"error": "Home not found"}

    avg_kwh = float(
        home["Energy Consumption (kWh)"].iloc[0]
    )

    cluster = int(home["cluster"].iloc[0])

    # =========================
    # FIND PACKAGE
    # =========================

    for pkg in SOLAR_PACKAGES:

        if avg_kwh <= pkg["max_kwh"]:

            return {

                "Home ID": home_id,

                "Cluster": cluster,

                "Avg Daily Energy (kWh)": round(avg_kwh, 2),

                "Recommended Package": pkg["name"],

                "Solar Panels": pkg["panels"],

                "Battery": pkg["battery"],

                "Inverter": pkg["inverter"],

                "Advice": [
                    "Use appliances during daylight hours",
                    "Shift heavy loads to solar peak time",
                    "Reduce peak-hour consumption"
                ]
            }

    return {
        "Home ID": home_id,
        "Package": "Custom High-Capacity Solar System",
        "Note": "Requires custom engineering design"
    }


# Example
print(recommend_solar(10))

{'Home ID': 10, 'Cluster': 2, 'Avg Daily Energy (kWh)': 1.49, 'Recommended Package': 'Small Home Kit', 'Solar Panels': '2 x 550W Solar Panels', 'Battery': '5 kWh Lithium Battery', 'Inverter': '2 kW Hybrid Inverter', 'Advice': ['Use appliances during daylight hours', 'Shift heavy loads to solar peak time', 'Reduce peak-hour consumption']}


In [16]:
print("\n" + "="*70)
print("SMART HOME → SOLAR RECOMMENDATION SYSTEM PIPELINE (DETAILED)")
print("="*70 + "\n")

pipeline = [

"📥 1. DATA LOADING",
"   - Download dataset using KaggleHub",
"   - Load CSV into Pandas DataFrame",
"   - Inspect columns: Home ID, Appliance Type, Energy, Time, Date",

"",

"🧹 2. DATA CLEANING",
"   - Remove missing values (NaN)",
"   - Remove duplicate records",
"   - Fix incorrect data types",
"   - Ensure consistent energy readings",

"",

"⏰ 3. FEATURE ENGINEERING (TIME + BEHAVIOR)",
"   - Convert Time → Hour (0–23)",
"   - Extract Date features (day, month, year)",
"   - Create Peak Hour flag (18–22)",
"   - Convert Season → numerical encoding",
"   - Convert Appliance Type → numerical encoding",

"",

"🏠 4. HOME-LEVEL AGGREGATION (USER PROFILING)",
"   - Group dataset by Home ID",
"   - Compute average energy consumption per home",
"   - Compute average temperature exposure",
"   - Compute household size impact",
"   - Compute appliance usage behavior",
"   - Result: one profile per home (user profile creation)",

"",

"📊 5. FEATURE MATRIX CREATION",
"   - Select features:",
"       • Avg Energy Consumption",
"       • Outdoor Temperature",
"       • Household Size",
"       • Hour usage pattern",
"       • Peak-hour behavior",
"       • Season behavior",
"       • Appliance usage pattern",

"",

"⚖️ 6. FEATURE SCALING",
"   - Apply StandardScaler",
"   - Normalize all features to same range",
"   - Prevent bias in clustering",

"",

"🧠 7. USER SEGMENTATION (MACHINE LEARNING)",
"   - Apply KMeans Clustering",
"   - Group similar energy behavior homes",
"   - Example clusters:",
"       • Cluster 0 → Low energy efficient homes",
"       • Cluster 1 → High AC usage homes",
"       • Cluster 2 → Night-heavy usage homes",
"       • Cluster 3 → High appliance load homes",

"",

"🔍 8. SIMILARITY ANALYSIS (COSINE SIMILARITY)",
"   - Compute similarity between homes",
"   - Find homes with similar consumption behavior",
"   - Identify energy-efficient vs inefficient peers",

"",

"☀️ 9. SOLAR SYSTEM SIZING ENGINE",
"   - Convert energy usage → solar requirement",
"   - Estimate daily kWh consumption per home",
"   - Map usage → system size",

"",

"🔋 10. COMPONENT RECOMMENDATION ENGINE",
"   - Recommend Solar Panels (Watt capacity)",
"   - Recommend Battery size (kWh storage)",
"   - Recommend Inverter size (kW load handling)",
"   - Match based on energy demand level",

"",

"💡 11. ENERGY OPTIMIZATION RECOMMENDATIONS",
"   - Suggest reducing peak-hour usage",
"   - Suggest appliance scheduling",
"   - Suggest switching to solar peak time usage",
"   - Suggest efficiency improvements",

"",

"🏆 12. TOP-N RECOMMENDATION GENERATION",
"   - Rank similar homes or solar packages",
"   - Return Top 5–10 recommendations",
"   - Include similarity + energy match score",

"",

"🌐 13. API LAYER (DEPLOYMENT)",
"   - Flask / FastAPI endpoint",
"   - Input: Home ID",
"   - Output: Solar system + recommendations JSON",

"",

"📊 14. DASHBOARD LAYER",
"   - Streamlit / React dashboard",
"   - Visualize energy usage patterns",
"   - Show cluster distribution",
"   - Show solar recommendations",
"   - Show cost & savings estimation",

"",

"🚀 FINAL OUTPUT",
"   → Solar Panel Recommendation",
"   → Battery Recommendation",
"   → Inverter Recommendation",
"   → Energy Saving Tips",
"   → Similar Home Comparison"
]

for step in pipeline:
    print(step)

print("\n" + "="*70)
print("END OF DETAILED PIPELINE")
print("="*70)


SMART HOME → SOLAR RECOMMENDATION SYSTEM PIPELINE (DETAILED)

📥 1. DATA LOADING
   - Download dataset using KaggleHub
   - Load CSV into Pandas DataFrame
   - Inspect columns: Home ID, Appliance Type, Energy, Time, Date

🧹 2. DATA CLEANING
   - Remove missing values (NaN)
   - Remove duplicate records
   - Fix incorrect data types
   - Ensure consistent energy readings

⏰ 3. FEATURE ENGINEERING (TIME + BEHAVIOR)
   - Convert Time → Hour (0–23)
   - Extract Date features (day, month, year)
   - Create Peak Hour flag (18–22)
   - Convert Season → numerical encoding
   - Convert Appliance Type → numerical encoding

🏠 4. HOME-LEVEL AGGREGATION (USER PROFILING)
   - Group dataset by Home ID
   - Compute average energy consumption per home
   - Compute average temperature exposure
   - Compute household size impact
   - Compute appliance usage behavior
   - Result: one profile per home (user profile creation)

📊 5. FEATURE MATRIX CREATION
   - Select features:
       • Avg Energy Consumptio

In [17]:
from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd

app = FastAPI()

home_df = pd.read_csv("home_profiles.csv")


SOLAR_PACKAGES = [
    {
        "name": "Small Kit",
        "max_kwh": 5,
        "panels": "2 x 550W",
        "battery": "5 kWh",
        "inverter": "2 kW"
    },
    {
        "name": "Medium Kit",
        "max_kwh": 10,
        "panels": "4 x 550W",
        "battery": "10 kWh",
        "inverter": "3 kW"
    },
    {
        "name": "Large Kit",
        "max_kwh": 20,
        "panels": "8 x 550W",
        "battery": "15 kWh",
        "inverter": "5 kW"
    }
]


class HomeRequest(BaseModel):
    home_id: int


@app.post("/recommend")
def recommend(req: HomeRequest):

    home = home_df[home_df["Home ID"] == req.home_id]

    if len(home) == 0:
        return {
            "status": "error",
            "message": "Home not found"
        }

    avg_kwh = float(home["Energy Consumption (kWh)"].iloc[0])
    cluster = int(home["cluster"].iloc[0])

    # solar mapping
    for pkg in SOLAR_PACKAGES:
        if avg_kwh <= pkg["max_kwh"]:
            return {
                "status": "success",

                "home_id": req.home_id,

                "energy_profile": {
                    "avg_daily_kwh": round(avg_kwh, 2),
                    "cluster": cluster
                },

                "recommendation": {
                    "package": pkg["name"],
                    "solar_panels": pkg["panels"],
                    "battery": pkg["battery"],
                    "inverter": pkg["inverter"]
                },

                "tips": [
                    "Use appliances during sunlight hours",
                    "Avoid peak-hour electricity usage",
                    "Shift heavy loads to daytime"
                ],

                "extra_info": {
                    "system_type": "Hybrid Solar Recommendation System",
                    "model_type": "KMeans + Rule-based Engine",
                    "data_source": "Smart Home Energy Dataset"
                }
            }

    return {
        "status": "success",
        "message": "Custom solar system required"
    }

In [18]:
{
  "status": "success",
  "home_id": 12,
  "energy_profile": {
    "avg_daily_kwh": 9.4,
    "cluster": 2
  },
  "recommendation": {
    "package": "Medium Kit",
    "solar_panels": "4 x 550W",
    "battery": "10 kWh",
    "inverter": "3 kW"
  },
  "tips": [
    "Use appliances during sunlight hours",
    "Avoid peak-hour electricity usage"
  ],
  "extra_info": {
    "system_type": "Hybrid Solar Recommendation System",
    "model_type": "KMeans + Rule-based Engine"
  }
}

{'status': 'success',
 'home_id': 12,
 'energy_profile': {'avg_daily_kwh': 9.4, 'cluster': 2},
 'recommendation': {'package': 'Medium Kit',
  'solar_panels': '4 x 550W',
  'battery': '10 kWh',
  'inverter': '3 kW'},
 'tips': ['Use appliances during sunlight hours',
  'Avoid peak-hour electricity usage'],
 'extra_info': {'system_type': 'Hybrid Solar Recommendation System',
  'model_type': 'KMeans + Rule-based Engine'}}